# Load and plot BorealScat-2 level 1.1 data
This notebook loads mode T3 data (multilooked tomograms from moving array measurements), extracts average intensity over a region of interest and plots the time series.

In [ ]:
import numpy as np
import pandas as pd 
from netCDF4 import Dataset
import xarray as xr
import h5py
import matplotlib.pyplot as plt
import os 
import glob
import matplotlib as mpl

## Get all tomogram file names

In [ ]:
datadir = '/data/s1/borealscat2/calibrated/2026' #data directory

#extract all file names
datadir_tomogs = os.path.join(datadir, 'tomog')
filenames = sorted(glob.glob(datadir_tomogs + '/bs2_radar_t3_1.1_*.h5'))
Nt = len(filenames)

## Extract datetimes (UTC and local solar time)

In [ ]:
def utc2lst(t):
    """
    Convert UTC to local solar time at Svartberget.
    """
    local_lon =  19.776; #local longitude, decimal degrees
    TC = np.timedelta64(4, 'm')*local_lon; #Difference between UTC and LST
    t += TC; #UTC to local solar time conversion

    return t

filename_meta = os.path.join(datadir, 'bs2_metadata.h5') #metadata file

with h5py.File(filename_meta, 'r') as file:
    t_str = [x.decode() for x in file['t']] #read string vector

t = pd.to_datetime(t_str, format='%Y%m%dt%H%M%S') #datestr to pandas datetime
t_lst = utc2lst(t) #UTC to local solar time
Nt =  t.size #number of time samples

## Extract 2D image coordinates

In [ ]:
with h5py.File(filename_meta, 'r') as fh:
    y_b1 = np.array(fh["band1/tomog_coords/y"])
    z_b1 = np.array(fh["band1/tomog_coords/z"])

    y_b2 = np.array(fh["band2/tomog_coords/y"])
    z_b2 = np.array(fh["band2/tomog_coords/z"])

    y_b3 = np.array(fh["band3/tomog_coords/y"])
    z_b3 = np.array(fh["band3/tomog_coords/z"])

## Load tomograms

### Preallocate Xarray DataArrays for tomograms

In [ ]:
dims = ("z", "y", "t")

coords = {"z":z_b1, "y":y_b1, "t":t_lst}
hh_band1 = xr.DataArray(dims=dims, coords=coords)
vv_band1 = xr.DataArray(dims=dims, coords=coords)
hv_band1 = xr.DataArray(dims=dims, coords=coords)
vh_band1 = xr.DataArray(dims=dims, coords=coords)

coords = {"z":z_b2, "y":y_b2, "t":t_lst}
hh_band2 = xr.DataArray(dims=dims, coords=coords)
vv_band2 = xr.DataArray(dims=dims, coords=coords)
hv_band2 = xr.DataArray(dims=dims, coords=coords)
vh_band2 = xr.DataArray(dims=dims, coords=coords)

coords = {"z":z_b3, "y":y_b3, "t":t_lst}
hh_band3 = xr.DataArray(dims=dims, coords=coords)
vv_band3 = xr.DataArray(dims=dims, coords=coords)
hv_band3 = xr.DataArray(dims=dims, coords=coords)
vh_band3 = xr.DataArray(dims=dims, coords=coords)

### Load tomograms from hdf5 files

In [ ]:
for tidx, fname in enumerate(filenames):

    with h5py.File(fname, 'r') as fh:
        #band 1
        hh_band1[:,:,tidx] = np.array(fh["band1/hh"])
        vv_band1[:,:,tidx] = np.array(fh["band1/vv"])
        hv_band1[:,:,tidx] = np.array(fh["band1/hv"])
        vh_band1[:,:,tidx] = np.array(fh["band1/vh"])

        #band 2
        hh_band2[:,:,tidx] = np.array(fh["band2/hh"])
        vv_band2[:,:,tidx] = np.array(fh["band2/vv"])
        hv_band2[:,:,tidx] = np.array(fh["band2/hv"])
        vh_band2[:,:,tidx] = np.array(fh["band2/vh"])

        #band 3
        hh_band3[:,:,tidx] = np.array(fh["band3/hh"])
        vv_band3[:,:,tidx] = np.array(fh["band3/vv"])
        hv_band3[:,:,tidx] = np.array(fh["band3/hv"])
        vh_band3[:,:,tidx] = np.array(fh["band3/vh"])

## Apply pixel weighting correction Q

### Load pixel weighting images

In [ ]:
dims = ("z", "y")

with h5py.File(filename_meta, 'r') as fh:
    
    #band 1
    y_Q_band1 = np.array(fh["band1/pixel_weight/y"])
    z_Q_band1 = np.array(fh["band1/pixel_weight/z"])
    coords = {"z":z_Q_band1, "y":y_Q_band1}
    
    Qhh_band1 = xr.DataArray(np.array(fh["band1/pixel_weight/hh"]), dims=dims, coords=coords)
    Qvv_band1 = xr.DataArray(np.array(fh["band1/pixel_weight/vv"]), dims=dims, coords=coords)
    Qhv_band1 = xr.DataArray(np.array(fh["band1/pixel_weight/hv"]), dims=dims, coords=coords)
    Qvh_band1 = xr.DataArray(np.array(fh["band1/pixel_weight/vh"]), dims=dims, coords=coords)

    #band 2
    y_Q_band2 = np.array(fh["band2/pixel_weight/y"])
    z_Q_band2 = np.array(fh["band2/pixel_weight/z"])
    coords = {"z":z_Q_band2, "y":y_Q_band2}

    Qhh_band2 = xr.DataArray(np.array(fh["band2/pixel_weight/hh"]), dims=dims, coords=coords)
    Qvv_band2 = xr.DataArray(np.array(fh["band2/pixel_weight/vv"]), dims=dims, coords=coords)
    Qhv_band2 = xr.DataArray(np.array(fh["band2/pixel_weight/hv"]), dims=dims, coords=coords)
    Qvh_band2 = xr.DataArray(np.array(fh["band2/pixel_weight/vh"]), dims=dims, coords=coords)
 
    #band 3
    y_Q_band3 = np.array(fh["band3/pixel_weight/y"])
    z_Q_band3 = np.array(fh["band3/pixel_weight/z"])
    coords = {"z":z_Q_band3, "y":y_Q_band3}

    Qhh_band3 = xr.DataArray(np.array(fh["band3/pixel_weight/hh"]), dims=dims, coords=coords)
    Qvv_band3 = xr.DataArray(np.array(fh["band3/pixel_weight/vv"]), dims=dims, coords=coords)
    Qhv_band3 = xr.DataArray(np.array(fh["band3/pixel_weight/hv"]), dims=dims, coords=coords)
    Qvh_band3 = xr.DataArray(np.array(fh["band3/pixel_weight/vh"]), dims=dims, coords=coords)

    

### Apply pixel weighting compensation

In [ ]:
#interpolate to match measured tomogram coordinates
Qhh_band1_int = Qhh_band1.interp(z=z_b1, y=y_b1)
Qvv_band1_int = Qvv_band1.interp(z=z_b1, y=y_b1)
Qhv_band1_int = Qhv_band1.interp(z=z_b1, y=y_b1)
Qvh_band1_int = Qvh_band1.interp(z=z_b1, y=y_b1)

Qhh_band2_int = Qhh_band2.interp(z=z_b2, y=y_b2)
Qvv_band2_int = Qvv_band2.interp(z=z_b2, y=y_b2)
Qhv_band2_int = Qhv_band2.interp(z=z_b2, y=y_b2)
Qvh_band2_int = Qvh_band2.interp(z=z_b2, y=y_b2)

Qhh_band3_int = Qhh_band3.interp(z=z_b3, y=y_b3)
Qvv_band3_int = Qvv_band3.interp(z=z_b3, y=y_b3)
Qhv_band3_int = Qhv_band3.interp(z=z_b3, y=y_b3)
Qvh_band3_int = Qvh_band3.interp(z=z_b3, y=y_b3)

#apply correction to get intensity image
I_hh_band1 = hh_band1/Qhh_band1_int
I_vv_band1 = vv_band1/Qvv_band1_int
I_hv_band1 = hv_band1/Qhv_band1_int
I_vh_band1 = vh_band1/Qvh_band1_int

I_hh_band2 = hh_band2/Qhh_band2_int
I_vv_band2 = vv_band2/Qvv_band2_int
I_hv_band2 = hv_band2/Qhv_band2_int
I_vh_band2 = vh_band2/Qvh_band2_int

I_hh_band3 = hh_band3/Qhh_band3_int
I_vv_band3 = vv_band3/Qvv_band3_int
I_hv_band3 = hv_band3/Qhv_band3_int
I_vh_band3 = vh_band3/Qvh_band3_int

### Plot tomograms with ROIs

In [ ]:
import matplotlib.patches as patches

def plot_tomog(data, t, ax, fig, vmin, title):
    datamax = data.max()
    s = (data-datamax).sel(t=tplot, method='nearest').plot(ax=ax, cmap='cubehelix', vmin=vmin, vmax=0, cbar_kwargs={'label':'Norm. intensity [dB]'})
    ax.set_title(title)
    ax.set_xlabel('Ground range [m]')
    ax.set_ylabel('Height [m]')

def plot_rois(zintervals, yintervals, ax, linestyle):
    groundrect = patches.Rectangle((yintervals[0], zintervals[0]), yintervals[1]-yintervals[0], zintervals[1]-zintervals[0] , linewidth=1.5, edgecolor='w', facecolor='none', linestyle=linestyle) # Create a Rectangle patch
    ax.add_patch(groundrect) # Add the patch to the Axes


mpl.rcParams['figure.dpi'] = 100

tplot = pd.to_datetime('20230701')
figsize = (7,6)

fig, axs = plt.subplots(nrows=3, ncols=1, figsize=figsize)

plot_tomog(10*np.log10(hh_band1), tplot, axs[0], fig, -40, 'P-band')
plot_tomog(10*np.log10(hh_band2), tplot, axs[1], fig, -40, 'UHF-band')
plot_tomog(10*np.log10(hh_band3), tplot, axs[2], fig, -50, 'L-band')

#plot_tomog(10*np.log10(vh_band1), tplot, axs[0], fig, -30, 'P-band')
#plot_tomog(10*np.log10(vh_band2), tplot, axs[1], fig, -30, 'UHF-band')
#plot_tomog(10*np.log10(vh_band3), tplot, axs[2], fig, -30, 'L-band')


plot_rois(zintervals=(-5,8), yintervals=(25,75), ax=axs[0], linestyle='--')
plot_rois(zintervals=(8,24), yintervals=(25,75), ax=axs[0], linestyle=':')

plot_rois(zintervals=(-5,6), yintervals=(25,75), ax=axs[1], linestyle='--')
plot_rois(zintervals=(6,24), yintervals=(25,75), ax=axs[1], linestyle=':')

plot_rois(zintervals=(-5,3), yintervals=(25,75), ax=axs[2], linestyle='--')
plot_rois(zintervals=(3,24), yintervals=(25,75), ax=axs[2], linestyle=':')

plt.tight_layout()
plt.show()

In [ ]:
mpl.rcParams['text.usetex'] = False
mpl.rcParams['figure.dpi'] = 100

def plot_tomog(data, t, ax, fig, vmin, title, add_colorbar):
    datamax = data.max()
    if add_colorbar==False:
        s = (data-datamax).sel(t=tplot, method='nearest').plot(ax=ax, cmap='magma', vmin=-25, vmax=0,  add_colorbar=False)
    else:
        s = (data-datamax).sel(t=tplot, method='nearest').plot(ax=ax, cmap='magma', vmin=-25, vmax=0, cbar_kwargs={'label':'Norm. intensity [dB]', 'location':None,'aspect':10})
    ax.set_title(title)
    ax.set_xlabel('Ground range [m]')
    ax.set_ylabel('Height [m]')



tplot = pd.to_datetime('20230701')
figsize = (8,2)

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=figsize)


(10*np.log10(Qhh_band3)).plot(ax=axs[0], cmap='gray', cbar_kwargs={'label':'Intensity [dB]','aspect':10})
#mpl.rcParams['text.usetex'] = True
axs[0].set(title='Space-variant \n system gain')
#mpl.rcParams['text.usetex'] = False
axs[0].set(ylabel='Height [m]', xlabel='Ground range [m]')
axs[0].set_xticks([25,50,75])
axs[0].set_yticks([-10,0,20])

plot_tomog(10*np.log10(hh_band3.sel(z=slice(z_Q_band3[0],z_Q_band3[-1]), y=slice(y_Q_band3[0],y_Q_band3[-1]) )), tplot, axs[1], fig, -30, 'Before \n compensation', True)
axs[1].set(ylabel='')
axs[1].set_xticks([25,50,75])
axs[1].set_yticks([-10,0,20])

plot_tomog(10*np.log10(I_hh_band3.sel(z=slice(z_Q_band3[0],z_Q_band3[-1]), y=slice(y_Q_band3[0],y_Q_band3[-1]) )), tplot, axs[2], fig, -30, 'After \n compensation', True)
axs[2].set(ylabel='')
axs[2].set_xticks([25,50,75])
axs[2].set_yticks([-10,0,20])

plt.tight_layout()
plt.show()



## Plot vertical profile time series

In [ ]:
figsize = (12,7)

mpl.rcParams['figure.dpi'] = 100
fig, axs = plt.subplots(nrows=4, ncols=1, figsize=figsize)
(10*np.log10(I_hh_band1.sel(y=slice(25,75), z=slice(-5,24)).mean(dim=["y"]))).plot(ax=axs[0], cmap='jet')
axs[0].set(ylabel='Height [m]')
axs[0].set(title='Band 1 (P-band), HH')

(10*np.log10(I_vv_band1.sel(y=slice(25,75), z=slice(-5,24)).mean(dim=["y"]))).plot(ax=axs[1], cmap='jet')
axs[1].set(ylabel='Height [m]')
axs[1].set(title='Band 1 (P-band), VV')

(10*np.log10(I_hv_band1.sel(y=slice(25,75), z=slice(-5,24)).mean(dim=["y"]))).plot(ax=axs[2], cmap='jet')
axs[2].set(ylabel='Height [m]')
axs[2].set(title='Band 1 (P-band), HV')

(10*np.log10(I_vh_band1.sel(y=slice(25,75), z=slice(-5,24)).mean(dim=["y"]))).plot(ax=axs[3], cmap='jet')
axs[3].set(ylabel='Height [m]')
axs[3].set(title='Band 1 (P-band), VH')

plt.tight_layout()
plt.show()

## Plot backscatter time series

In [ ]:
figsize = (16,9)
linewidth=0.5
region = 'canopy'


mpl.rcParams['figure.dpi'] = 80

if region == 'ground':
    zmin_p, zmax_p = -5,8 #ground
    zmin_uhf, zmax_uhf = -5,6
    zmin_l, zmax_l = -5,3
if region == 'canopy':
    zmin_p, zmax_p = 8,24 #canopy
    zmin_uhf, zmax_uhf = 6,24
    zmin_l, zmax_l = 3,24
if region == 'full forest':
    zmin_p, zmax_p = 8,24 #canopy
    zmin_uhf, zmax_uhf = -5,24
    zmin_l, zmax_l = -5,24

#P-band
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=figsize)
fig.suptitle('Band 1 (P-band) backscatter [dB]')
(10*np.log10(I_hh_band1.sel(y=slice(25,75), z=slice(zmin_p,zmax_p)).mean(dim=["y","z"]))).plot(ax=axs[0], linewidth=linewidth)
(10*np.log10(I_vv_band1.sel(y=slice(25,75), z=slice(zmin_p,zmax_p)).mean(dim=["y","z"]))).plot(ax=axs[1], linewidth=linewidth)
(10*np.log10(I_hv_band1.sel(y=slice(25,75), z=slice(zmin_p,zmax_p)).mean(dim=["y","z"]))).plot(ax=axs[2], linewidth=linewidth)
(10*np.log10(I_vh_band1.sel(y=slice(25,75), z=slice(zmin_p,zmax_p)).mean(dim=["y","z"]))).plot(ax=axs[2], linewidth=linewidth)
axs[0].set_title('HH')
axs[1].set_title('VV')
axs[2].set_title('Cross pol')
axs[0].grid()
axs[1].grid()
axs[2].grid()
plt.tight_layout()
plt.show()


#UHF-band
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=figsize)
fig.suptitle('Band 2 (UHF-band) backscatter [dB]')
(10*np.log10(I_hh_band2.sel(y=slice(25,75), z=slice(zmin_uhf,zmax_uhf)).mean(dim=["y","z"]))).plot(ax=axs[0], linewidth=linewidth)
(10*np.log10(I_vv_band2.sel(y=slice(25,75), z=slice(zmin_uhf,zmax_uhf)).mean(dim=["y","z"]))).plot(ax=axs[1], linewidth=linewidth)
(10*np.log10(I_hv_band2.sel(y=slice(25,75), z=slice(zmin_uhf,zmax_uhf)).mean(dim=["y","z"]))).plot(ax=axs[2], linewidth=linewidth)
(10*np.log10(I_vh_band2.sel(y=slice(25,75), z=slice(zmin_uhf,zmax_uhf)).mean(dim=["y","z"]))).plot(ax=axs[2], linewidth=linewidth)
axs[0].set_title('HH')
axs[1].set_title('VV')
axs[2].set_title('Cross pol')
axs[0].grid()
axs[1].grid()
axs[2].grid()
plt.tight_layout()
plt.show()

#L-band
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=figsize)
fig.suptitle('Band 3 (L-band) backscatter [dB]')
(10*np.log10(I_hh_band3.sel(y=slice(25,75), z=slice(zmin_l,zmax_l)).mean(dim=["y","z"]))).plot(ax=axs[0], linewidth=linewidth)
(10*np.log10(I_vv_band3.sel(y=slice(25,75), z=slice(zmin_l,zmax_l)).mean(dim=["y","z"]))).plot(ax=axs[1], linewidth=linewidth)
(10*np.log10(I_hv_band3.sel(y=slice(25,75), z=slice(zmin_l,zmax_l)).mean(dim=["y","z"]))).plot(ax=axs[2], linewidth=linewidth)
(10*np.log10(I_vh_band3.sel(y=slice(25,75), z=slice(zmin_l,zmax_l)).mean(dim=["y","z"]))).plot(ax=axs[2], linewidth=linewidth)
axs[0].set_title('HH')
axs[1].set_title('VV')
axs[2].set_title('Cross pol')
axs[0].grid()
axs[1].grid()
axs[2].grid()
#axs[0].set_ylim(81,85)
#axs[1].set_ylim(77,81)
plt.tight_layout()
plt.show()
